In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle

import structlog
import logging
structlog.configure(
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
)

import sys
sys.path.append('../../../../')


from src.difsched.agents.dr3rlpy import train_bc, from_env_to_d3rlpy_dataset, evaluate
from src.difsched.agents.gym_env import HybridEnv
from src.difsched.config import getExpConfig, visualizeExpConfig
from src.difsched.env.Hybrid import createEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\Ye\miniconda3\envs\traffic_predictor_3_9\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
offlinDatasetFolder = f'../../../../data/processed/offline_dataset'
trafficDatasetFolder = f'../../../../data/processed/traffic'
outputFolder = f'../../../../data/processed/d3rlpy'

In [ ]:
for expConfigIdx in range(24):
    expParams = getExpConfig(expConfigIdx)
    visualizeExpConfig(expParams)

    # ============== Load offline dataset ==============
    dataset_off = {
        'observations': [],
        'actions': [],
        'rewards': [],
        'next_observations': []
    }

    for exp_idx in expParams['offline_dataset_idxs']:
        with open(f'{offlinDatasetFolder}/subOptimalAgent_envConfig{exp_idx}.pkl', 'rb') as f:
            dataset_expert = pickle.load(f)
        
        dataset_off['observations'].extend(dataset_expert['uRecord'])
        dataset_off['actions'].extend(dataset_expert['actionsRecord'])
        dataset_off['rewards'].extend(dataset_expert['rewardRecord'])
        dataset_off['next_observations'].extend(dataset_expert['uNextRecord'])
        
        print(f"Exp {exp_idx} - Avg. packet loss rate: {np.mean(dataset_expert['rewardRecord'])}")
        print(f"Exp {exp_idx} - length of dataset: {len(dataset_expert['uRecord'])}")

    print(f"\nCombined dataset length: {len(dataset_off['observations'])}")
    print(f"Combined avg. packet loss rate: {np.mean(dataset_off['rewards'])}")

    # ============== Convert to d3rlpy dataset ==============
    dataset = from_env_to_d3rlpy_dataset(dataset_off, expParams)
    print(f"Dataset created successfully!")
    print(f"Number of episodes: {dataset.size()}")

    ep = list(dataset.episodes)[0]

    print("="*50)
    print("Offline Dataset Episode Visualization")
    print("="*50)

    # Number of timesteps in this episode
    n_steps = len(ep.actions[0])
    print(f"Number of Timesteps:        {n_steps}")

    # Observation and Action dimensions
    obs_dim = len(ep.observations[0]) if isinstance(ep.observations[0], (list, np.ndarray)) else 1
    action_dim = len(ep.actions[0]) if isinstance(ep.actions[0], (list, np.ndarray)) else 1
    print(f"Observation Dimension:      {obs_dim}")
    print(f"Action Dimension:           {action_dim}")

    # Range for actions and observations, flatten if possible
    obs_array = np.array(ep.observations[0])
    act_array = np.array(ep.actions[0])
    obs_min = np.min(obs_array.flatten())
    obs_max = np.max(obs_array.flatten())
    act_min = np.min(act_array.flatten())
    act_max = np.max(act_array.flatten())
    print(f"Observation Range:          min={obs_min} max={obs_max}")
    print(f"Action Range:               min={act_min} max={act_max}")

    # Reward statistics
    mean_reward = np.mean(ep.rewards)
    print(f"Mean Reward:                {mean_reward:.4f}")

    print("="*50)

    # ============== Save dataset ==============
    # Save dataset as a pickle file
    dataset_save_path = f"{outputFolder}/d3rlpy_dataset_exp{expConfigIdx}.pkl"
    with open(dataset_save_path, "wb") as f:
        pickle.dump(dataset, f)
    print(f"Dataset saved to {dataset_save_path}")

    # Read the dataset back from the pickle file
    with open(dataset_save_path, "rb") as f:
        loaded_dataset = pickle.load(f)
    print("Dataset loaded from pickle file successfully.")
    print(f"Loaded dataset size: {loaded_dataset.size()}")

EnvType: HYBRID
N_user: 8
LEN_window: 20
N_aggregation: 4
dataflow: haptic_1ms_20
randomSeed: 999
r_bar: 4
B: 100
sigma_list: [0.7, 0.75, 0.8, 0.85, 0.9]
offline_dataset_idxs: [0, 1, 2]
Exp 0 - Avg. packet loss rate: 0.007220242274743211
Exp 0 - length of dataset: 10000
Exp 1 - Avg. packet loss rate: 0.006806147144592677
Exp 1 - length of dataset: 10000
Exp 2 - Avg. packet loss rate: 0.006796585836089212
Exp 2 - length of dataset: 10000

Combined dataset length: 30000
Combined avg. packet loss rate: 0.006940991751808366
Observations shape: (30000, 8)
Number of episodes: 30
Dataset created successfully!
Number of episodes: 30
Offline Dataset Episode Visualization
Number of Timesteps:        18
Observation Dimension:      8
Action Dimension:           18
Observation Range:          min=0.05000000074505806 max=0.10000000149011612
Action Range:               min=-1.0 max=-0.1111111119389534
Mean Reward:                0.9922
Dataset saved to ../../../../data/processed/d3rlpy/d3rlpy_dataset